# VayuGNN — Neighborhood Distribution Grid Brain

Trains a GNN that ingests 12 historical graph snapshots and predicts:
- **Overload probability** (next 30 min, per-minute)
- **Voltage forecast** (next 30 min, per-minute)
- **Neighborhood risk** (next 30 min, single scalar)
- **Duck curve** (net load, next 24 h at 15-min resolution)

Run with **dry-run** (random data, no files needed) or **real data** (Pecan Street + NSRDB).

## 1. Setup

Choose whether to use real data (requires uploading to Drive) or dry-run.

In [ ]:
# ─── Configuration ───
USE_REAL_DATA = False   # set True to train on Pecan Street + NSRDB data
DATA_CITY = "bangalore"  # bangalore | chennai | delhi | hyderabad | kochi
EPOCHS = 50
NUM_EPISODES = 10       # simulator episodes for dataset generation
BATCH_SIZE = 4
CHECKPOINT_DIR = "/content/drive/MyDrive/vayugrid_checkpoints" if USE_REAL_DATA else "checkpoints"

if USE_REAL_DATA:
    from google.colab import drive
    drive.mount("/content/drive")
    print("Drive mounted. Place data at: /content/drive/MyDrive/vayugrid_data/")
    print("  - data/processed/pecan_india/{city}/2019/pecan_wired_{city}_2019.csv")
    print("  - data/nsrdb_himawari/{city}/{city}_2019.csv")

In [ ]:
# ─── Install dependencies ───
!pip install -q gymnasium onnx onnxruntime 'pandas<3' 'numpy<2'

In [ ]:
# ─── Clone repo & install package ───
import os, sys
REPO_DIR = "/content/vayugrid"
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/V4RSH1TH-R3DDY/VayuGrid.git {REPO_DIR}
%cd {REPO_DIR}

if USE_REAL_DATA:
    # Symlink data from Drive
    DRIVE_DATA = "/content/drive/MyDrive/vayugrid_data"
    if not os.path.exists("data"):
        !ln -s "{DRIVE_DATA}/data" data
        print(f"Linked data from {DRIVE_DATA}/data")

# Install in editable mode (skip torch — Colab has it)
!pip install -e . --no-deps -q
print("Repo ready")

## 2. Training

Generates the dataset (or uses dummy data) and trains the GNN.

In [ ]:
import torch
from pathlib import Path
from torch.utils.data import DataLoader, Dataset
from typing import cast, Any

from ai.gnn import (
    VayuGNN, VayuGNNTrainer, VayuGNNLoss, GNNTrainConfig,
    GNNSample, GraphDatasetGenerator,
)
from simulator.config import load_simulator_config

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

checkpoint_dir = Path(CHECKPOINT_DIR)
checkpoint_dir.mkdir(parents=True, exist_ok=True)

# ── Model setup ──
model = VayuGNN().to(device)
loss_fn = VayuGNNLoss()
train_cfg = GNNTrainConfig(
    lr=1e-3, epochs=EPOCHS,
    checkpoint_path=str(checkpoint_dir / "vayu_gnn_best.pt"),
)
trainer = VayuGNNTrainer(model=model, cfg=train_cfg, loss_fn=loss_fn, device=device)

# ── Collate ──
def collate_fn(batch: list[GNNSample]) -> tuple[list[list[Any]], dict[str, torch.Tensor]]:
    snap_seqs = [s.snapshots for s in batch]
    targets = {
        "target_overload": torch.stack([s.target_overload for s in batch]),
        "target_voltage":  torch.stack([s.target_voltage for s in batch]),
        "target_risk":     torch.stack([s.target_risk for s in batch]),
        "target_duck":     torch.stack([s.target_duck for s in batch]),
    }
    return snap_seqs, targets

# ── Data ──
if not USE_REAL_DATA:
    print("Dry-run mode: using dummy random data")
    from ai.gnn import _make_dummy_snapshots
    dummy = [
        GNNSample(
            snapshots=_make_dummy_snapshots(),
            target_overload=torch.rand(30),
            target_voltage=torch.rand(30),
            target_risk=torch.rand(1),
            target_duck=torch.rand(96),
        )
        for _ in range(32)
    ]
    train_loader = DataLoader(cast(Dataset, dummy), batch_size=4, shuffle=True, collate_fn=collate_fn)
    val_loader   = DataLoader(cast(Dataset, dummy), batch_size=4, collate_fn=collate_fn)
else:
    print(f"Generating dataset from {NUM_EPISODES} simulator episodes...")
    scenario_path = "scenarios/phase1_default.json"
    sim_cfg = load_simulator_config(scenario_path)
    sim_cfg.load_profile.use_pecan_profiles = True
    sim_cfg.load_profile.pecan_profile_file = (
        f"data/processed/pecan_india/{DATA_CITY}/2019/pecan_wired_{DATA_CITY}_2019.csv"
    )
    sim_cfg.load_profile.city = DATA_CITY

    generator = GraphDatasetGenerator(sim_cfg)
    train_samples, val_samples, _ = generator.generate_dataset(num_episodes=NUM_EPISODES)
    print(f"  Train: {len(train_samples)} | Val: {len(val_samples)}")

    train_loader = DataLoader(
        cast(Dataset, train_samples), batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn
    )
    val_loader = DataLoader(
        cast(Dataset, val_samples), batch_size=BATCH_SIZE, collate_fn=collate_fn
    )

# ── Train ──
print(f"\nTraining for {EPOCHS} epochs...")
trainer.fit(train_loader, val_loader)
print(f"\nCheckpoint saved to: {train_cfg.checkpoint_path}")

In [ ]:
# ─── False-positive rate (only meaningful with real data) ───
if USE_REAL_DATA:
    fpr = trainer.compute_false_positive_rate(val_loader)
    print(f"False-positive rate: {fpr*100:.2f}% (gate: <1%)")
else:
    print("Skipping FPR computation (dry-run data has no signal)")

## 3. Download Checkpoint

Download the trained model for deployment.

In [ ]:
from google.colab import files
if Path(train_cfg.checkpoint_path).exists():
    files.download(train_cfg.checkpoint_path)
    print("Downloaded!")
else:
    print("No checkpoint found yet — train first.")